Generating files in Volume - simulating 1minute candles in market
One 'syntetic candle' contains data about:
- Open price
- Close price
- Highest price
- Lowest price
- Timestamp

In [0]:
dbutils.widgets.text("catalog_name", "dbr_dev")
dbutils.widgets.text("container", "trezio2005")
dbutils.widgets.text("storage_account", "dlspl21databricks")
dbutils.widgets.text("volume_name", "raw_data")

catalog_name = dbutils.widgets.get("catalog_name")
container = dbutils.widgets.get("container")
storage_account = dbutils.widgets.get("storage_account")
volume_name = dbutils.widgets.get("volume_name")

volume_path = f"/Volumes/{catalog_name}/{container}_bronze/{volume_name}/"

Generating files - 1440 1m candles = 1 day

In [0]:
import time
from datetime import datetime, timedelta
import random
import json
import os

current_timestamp = datetime.now().replace(hour=0, minute=0, second=0, microsecond=0)

#simulating the market
current_price = 150.0  
volatility = 0.5

def generate_sample_data(timestamp, open_price):
    price_change = random.uniform(-volatility, volatility)
    close_price = round(open_price + price_change, 2)
    highest_price = round(max(open_price, close_price) + random.uniform(0, volatility/2), 2)
    lowest_price = round(min(open_price, close_price) - random.uniform(0, volatility/2), 2)

    data = {
        "Open price": open_price,
        "Close price": close_price,
        "Highest price": highest_price,
        "Lowest price": lowest_price,
        "Timestamp": timestamp.isoformat()
    }

    return data

for i in range(1, 1441):
    current_timestamp = current_timestamp + timedelta(minutes=1)
    file_name = f"1m_candle_{current_timestamp.strftime('%Y-%m-%d_%H-%M-%S')}.json"
    file_path = os.path.join(volume_path, file_name)

    sample_data = generate_sample_data(current_timestamp, current_price)
    current_price = sample_data["Close price"]

    with open(file_path, "w") as file:
        json.dump(sample_data, file)


Generating additional files with changed columns. Added one more column with Volume

In [0]:
for i in range(1, 6):
    current_timestamp = current_timestamp + timedelta(minutes=1)
    file_name = f"1m_candle_{current_timestamp.strftime('%Y-%m-%d_%H-%M-%S')}.json"
    file_path = os.path.join(volume_path, file_name)

    sample_data = generate_sample_data(current_timestamp, current_price)
    current_price = sample_data["Close price"]

    sample_data["Volume"] = random.randint(100, 5000)
    
    with open(file_path, "w") as file:
        json.dump(sample_data, file)
